# 01 — Inspección Inicial

**Objetivo:** Solo *mirar* el dataset crudo. Cargamos el JSON y exploramos con `.head()`, `.info()`, `.describe()` y conteos. Anotamos todo lo que está mal, **pero no corregimos nada todavía** (eso es trabajo del notebook 02).

> Regla de esta etapa: observar y documentar. Cada anomalía detectada acá se convierte en una tarea de limpieza en el notebook siguiente.

## Carga del dataset crudo

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

# Cargamos el JSON original (crudo, sin tocar)
df = pd.read_json('../data/raw/streaming_users_dirty.json')
print("Dimensiones:", df.shape)
df.head()

Dimensiones: (8160, 8)


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


## Clasificación de las variables según su tipo

Antes de cualquier análisis, identificamos el **tipo de cada variable**, ya que esto determina qué
estadísticos y gráficos son apropiados (medias y desvíos para numéricas; frecuencias y moda para
categóricas). Seguimos la clasificación: **cualitativas** (nominales / ordinales) y **cuantitativas**
(discretas / continuas).

In [2]:
clasificacion = pd.DataFrame({
    'variable': ['user_id', 'age', 'subscription_plan', 'monthly_watch_time_mins',
                 'country', 'favorite_genre', 'last_login_date', 'customer_support_tickets'],
    'tipo': ['Identificador', 'Cuantitativa discreta', 'Cualitativa ordinal',
             'Cuantitativa continua', 'Cualitativa nominal', 'Cualitativa nominal',
             'Fecha (temporal)', 'Cuantitativa discreta'],
    'descripcion': [
        'Clave única del usuario (no se analiza estadísticamente)',
        'Edad en años',
        'Plan contratado (Básico < Estándar < Premium: tiene orden)',
        'Minutos de visualización por mes',
        'País de residencia',
        'Género de contenido preferido',
        'Fecha del último acceso',
        'Cantidad de tickets de soporte abiertos',
    ]
})
clasificacion

,variable,tipo,descripcion
0,user_id,Identificador,Clave única del usuario (no se analiza estadís...
1,age,Cuantitativa discreta,Edad en años
2,subscription_plan,Cualitativa ordinal,Plan contratado (Básico < Estándar < Premium: ...
3,monthly_watch_time_mins,Cuantitativa continua,Minutos de visualización por mes
4,country,Cualitativa nominal,País de residencia
5,favorite_genre,Cualitativa nominal,Género de contenido preferido
6,last_login_date,Fecha (temporal),Fecha del último acceso
7,customer_support_tickets,Cuantitativa discreta,Cantidad de tickets de soporte abiertos


**Observaciones sobre los tipos:**
- `subscription_plan` es **ordinal**: sus categorías tienen un orden natural (Básico < Estándar < Premium), a diferencia de `country` o `favorite_genre`, que son **nominales** (sin orden).
- `age` y `customer_support_tickets` son **discretas** (conteos enteros); `monthly_watch_time_mins` es **continua**.
- `user_id` es un identificador: sirve para detectar duplicados, pero no se analiza como variable estadística.
- `last_login_date` es temporal: requiere parseo a tipo fecha antes de poder usarse.

## Estructura y tipos de dato

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB


**Observación:** El dataset tiene 8 columnas. Conviene revisar si los tipos son los esperados (por ejemplo, `last_login_date` aparece como texto/objeto en vez de fecha, lo que ya anticipa problemas de formato).

## Estadísticas descriptivas

In [4]:
df.describe().round(2)

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


**Anomalías que saltan en el `describe()`:**
- **`age`**: el mínimo y el máximo son imposibles (hay valores como **-5** y **150**). Una edad no puede ser negativa ni superar ~100.
- **`monthly_watch_time_mins`**: hay **valores negativos** (mirar tiempo negativo no existe) y un máximo absurdo (**99999**), muy lejos del resto.
- **`customer_support_tickets`**: aparece **-1** (imposible) y saltos a **99 / 150**, desconectados del resto de los valores.

## Detección de valores faltantes

In [5]:
faltantes = df.isnull().sum()
pct = (faltantes / len(df) * 100).round(2)
pd.DataFrame({'faltantes': faltantes, '%': pct}).query('faltantes > 0')

,faltantes,%
monthly_watch_time_mins,193,2.37
favorite_genre,240,2.94
last_login_date,320,3.92


**Observación:** Hay nulos en `monthly_watch_time_mins`, `favorite_genre` y `last_login_date`. Ninguno es masivo (todos por debajo del ~4%), así que en el notebook 02 conviene **imputar** en vez de borrar filas.

## Detección de duplicados

In [6]:
print("user_id duplicados:", df['user_id'].duplicated().sum())
print("Filas 100% idénticas:", df.duplicated().sum())

user_id duplicados: 160
Filas 100% idénticas: 126


**Observación:** `user_id` debería ser único pero hay **160 repetidos**. Habrá que eliminarlos en la limpieza.

## Inspección de variables categóricas

In [7]:
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"=== {col} ({df[col].nunique(dropna=True)} valores distintos) ===")
    print(df[col].value_counts(dropna=False))
    print()

=== subscription_plan (15 valores distintos) ===
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

=== country (26 valores distintos) ===
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64

=== favorite_genre (28 valores distintos) ===
favorite_genre
Comedia

**Anomalías de codificación detectadas:**
- **`subscription_plan`**: el mismo plan aparece escrito de muchas formas — `Básico`/`basico`/`BASICO`/`Basic`, `Estándar`/`Std`/`estandar`/`STANDARD`, `Premium`/`Premiun` (con error de tipeo). Sin unificar, se cuentan como categorías distintas.
- **`country`**: mezcla de nombres completos y códigos/idiomas (`Argentina`/`ARG`/`argentina`, `Brasil`/`Brazil`/`BRA`, `México`/`MEX`/`méxico`...).
- **`favorite_genre`**: mismo problema (`Acción`/`accion`/`Action`, `thriler` mal escrito, `comedy`/`Comedia`, `Crimen`/`crime`...).

## Inspección de la columna de fechas

In [8]:
# Intentamos parsear para ver cuántas fechas son inválidas
fechas = pd.to_datetime(df['last_login_date'], errors='coerce', format='mixed', dayfirst=False)
invalidas = fechas.isna() & df['last_login_date'].notna()
print("Valores de fecha no parseables:", invalidas.sum())
print("\nEjemplos de valores problemáticos:")
print(df.loc[invalidas, 'last_login_date'].value_counts().head(10))

Valores de fecha no parseables: 64

Ejemplos de valores problemáticos:
last_login_date
2026-15-03       20
0000-00-00       17
not_available    14
31-02-2022       13
Name: count, dtype: int64


**Anomalías detectadas en `last_login_date`:**
- Formatos mezclados (`2023/12/28`, `31-02-2022`, `11-19-2018`).
- Basura literal: `not_available`, `0000-00-00`.
- Fechas imposibles: `2026-15-03` (mes 15), días inexistentes (`31-02`).
- Posibles fechas futuras que no pueden ser un "último login".

## Resumen de problemas detectados (checklist para el notebook 02)

| # | Variable | Problema | Acción propuesta (notebook 02) |
|---|----------|----------|-------------------------------|
| 1 | `user_id` | 160 duplicados | Eliminar duplicados |
| 2 | `age` | Valores imposibles (-5, 0, 130, 150) | Imposibles → nulo → imputar |
| 3 | `subscription_plan` | Múltiples grafías del mismo plan | Normalizar a 3 categorías |
| 4 | `country` | Nombres/códigos/idiomas mezclados | Normalizar a 7 países |
| 5 | `favorite_genre` | Grafías mezcladas + 240 nulos | Normalizar + imputar |
| 6 | `monthly_watch_time_mins` | 193 nulos, negativos, valores imposibles (99999) | Inválidos → nulo, winsorizar cola real, imputar |
| 7 | `customer_support_tickets` | -1 y valores imposibles (99, 150) | Inválidos → nulo → imputar |
| 8 | `last_login_date` | Formatos mixtos, basura, fechas imposibles/futuras | Parsear; inválidas → nulo |

**Nada se modificó en este notebook.** Solo observamos y documentamos. La limpieza ocurre en `02_calidad_y_limpieza.ipynb`.